# AksharaPiece on Kaggle: training Transformer and Mamba models

This notebook tests the main research question: **do Nepali language models learn better when their tokenizer keeps every syllable (akshara) whole?**

It trains four small language models (about 27M parameters each) on **exactly the same Nepali text**:

| | Plain BPE 16k | AksharaPiece BPE 16k |
|---|---|---|
| **Transformer** (8 layers) | run 1 | run 2 |
| **Mamba-2** (16 layers) | run 3 | run 4 |

and compares them with **bits per byte (BPB)** on held-out text: how many bits the model needs per byte of Nepali text. Lower is better, and unlike perplexity it is fair across tokenizers.

**Before you save and run**

1. **Turn on the GPU.** In the right-hand panel open **Session options → Accelerator → GPU T4 x2**. Kaggle needs a verified phone number for GPUs. Two GPUs let two models train at once (P100 also works, about twice as slow).
2. **Internet → On.**
3. **Hugging Face token** (recommended, avoids slow or stalled downloads): **Add-ons → Secrets**, make sure `HF_TOKEN` is listed and switched on for this notebook.
4. **Check it took effect:** run the first three code cells by hand. The setup cell must print `GPUs: ['Tesla T4', 'Tesla T4']`. If it prints `none`, the notebook stops on purpose: fix the accelerator first.
5. Then **Save Version → Save & Run All (Commit)**. The version page should show **Accelerator: GPU T4 x2**. Stop the editor session afterwards so it does not use GPU hours while idle.

Run time: roughly **2–3 hours** on T4 x2 (about 5 of your 30 free GPU hours per week; two GPUs count double). For a quick test, set `TRAIN_MB = 50` (about 20 minutes).

In [ ]:
# ---- Settings ----
RAW_MB = 1500         # Nepali text to download from IndicCorp v2 (MB); the file is 14.8 GB in total
HELDOUT_MB = 20       # held-out text, never used for training
TOKENIZER_MB = 100    # text for training the tokenizers
TRAIN_MB = 600        # text each language model trains on (identical bytes for every run)
VAL_MB = 5            # held-out text used to compute bits per byte
VOCABS = [16000]      # add 8000 to also compare at 8k (adds 4 more runs)
ARCHS = ["mamba", "transformer"]
SEQ_LEN = 1024
SEED = 0
ALLOW_CPU = False     # keep False: without a GPU the notebook stops instead of running for days

## Setup
Unpacks the project code (embedded in this notebook) and checks the GPUs.

In [ ]:
import base64, io, json, os, subprocess, sys, time, zipfile

ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = "/kaggle/working" if ON_KAGGLE else os.getcwd()          # saved by Kaggle: keep it small
WORK = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else ("/tmp/aksharapiece_work" if ON_KAGGLE else os.getcwd())
os.makedirs(WORK, exist_ok=True)                                # large data lives here, not saved as output
RUNS_OUT = os.path.join(OUT, "finished_runs")                   # each finished run is copied here at once
os.makedirs(RUNS_OUT, exist_ok=True)
os.chdir(WORK)
CODE_ZIP_B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_ZIP_B64))).extractall(WORK)   # always refresh the code
PROJECT = os.path.join(WORK, "aksharapiece")
os.chdir(PROJECT)
for p in (PROJECT, os.path.join(PROJECT, "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONPATH"] = PROJECT


def run(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed: {cmd}")


import torch
N_GPU = torch.cuda.device_count()
print("PyTorch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)] or "none")
print("large data in", WORK, "| saved output in", OUT)
if N_GPU == 0 and not ALLOW_CPU:
    raise RuntimeError("No GPU found. Open Session options -> Accelerator -> GPU T4 x2, then run again. "
                       "(Stopping now so the run does not take days on the CPU.)")

In [ ]:
run("python -m pytest -q tests")

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in to Hugging Face.")
except Exception:
    print("No HF_TOKEN secret: downloading without a token (may be slower).")

## Step 1. Download and clean Nepali text
Streams the first `RAW_MB` of the IndicCorp v2 Nepali file (CC-0), then normalizes, filters, deduplicates and splits it into held-out, tokenizer and pretraining text.

In [ ]:
import requests


def download_prefix(url, dest, limit, token=None, attempts=8):
    """Download the first `limit` bytes of a large file with HTTP range requests.
    Prints progress every 100 MB, resumes after a stall or dropped connection, and ends the file
    at a complete line."""
    size = os.path.getsize(dest) if os.path.exists(dest) else 0
    t0, next_report = time.time(), (size // 100_000_000 + 1) * 100_000_000
    for attempt in range(1, attempts + 1):
        if size >= limit:
            break
        headers = {"Range": f"bytes={size}-{limit - 1}"}
        if token:
            headers["Authorization"] = f"Bearer {token}"
        try:
            with requests.get(url, headers=headers, stream=True, timeout=(30, 120)) as r:
                if r.status_code not in (200, 206):
                    raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
                if r.status_code == 200 and size:      # server ignored the range: start over
                    size = 0
                with open(dest, "ab" if size else "wb") as out:
                    for chunk in r.iter_content(8 * 1024 * 1024):
                        out.write(chunk)
                        size += len(chunk)
                        if size >= next_report:
                            mb_s = size / 1e6 / max(1e-9, time.time() - t0)
                            print(f"  {size / 1e6:,.0f} MB downloaded ({mb_s:.1f} MB/s)", flush=True)
                            next_report += 100_000_000
                        if size >= limit:
                            break
        except Exception as e:
            print(f"  attempt {attempt}: {type(e).__name__}: {str(e)[:150]} - resuming in 10 s", flush=True)
            time.sleep(10)
    if size < limit * 0.99:
        raise RuntimeError(f"download incomplete: {size / 1e6:.0f} MB of {limit / 1e6:.0f} MB")
    with open(dest, "rb+") as f:                        # cut to the last complete line
        f.seek(max(0, size - 1_000_000))
        tail_bytes = f.read()
        f.truncate(size - len(tail_bytes) + tail_bytes.rfind(b"\n") + 1)
    print(f"saved {os.path.getsize(dest) / 1e6:,.0f} MB in {time.time() - t0:.0f}s", flush=True)


RAW_FILE = "data/raw/indiccorp_ne_sample.txt"
os.makedirs("data/raw", exist_ok=True)
if not (os.path.exists(RAW_FILE) and os.path.getsize(RAW_FILE) >= RAW_MB * 0.99e6):
    from huggingface_hub import get_token, hf_hub_url
    download_prefix(hf_hub_url("ai4bharat/IndicCorpV2", "data/ne.txt", repo_type="dataset"),
                    RAW_FILE, RAW_MB * 1_000_000, token=get_token())

if not os.path.exists("data/clean/stats.json"):
    run(f"python scripts/clean.py --inputs {RAW_FILE} --out data/clean --heldout-mb {HELDOUT_MB} "
        f"--tok-mb {TOKENIZER_MB} --pretrain-mb 100000")
stats = json.load(open("data/clean/stats.json"))
print(f"kept {stats['kept']:,} lines; MB written: {stats['written_mb']}")
assert stats["written_mb"]["pretrain.txt"] >= TRAIN_MB * 0.98, "not enough pretraining text: raise RAW_MB or lower TRAIN_MB"

## Step 2. Train the tokenizers
Plain BPE and AksharaPiece BPE, same trainer, same text, same vocabulary size. AksharaPiece keeps the syllables that cover 99.9% of occurrences as whole symbols (99.5% below 16k).

In [ ]:
run(f"python scripts/train_tokenizers.py --train-file data/clean/tokenizer_train.txt --train-mb {TOKENIZER_MB} "
    f"--vocab {' '.join(map(str, VOCABS))} --out data/tok")

## Step 3. Tokenize the training and validation text
Every tokenizer encodes the identical bytes. The table shows how many tokens each needs.

In [ ]:
TOKS = []
for v in VOCABS:
    k = f"bpe_{v // 1000}k"
    TOKS += [(f"plain_{k}", f"hf:data/tok/plain_{k}.json"),
             (f"akshara_{k}", f"aksharapiece:data/tok/akshara_{k}.aksharapiece.json")]
tok_args = " ".join(f"--tok {n}={s}" for n, s in TOKS)
run(f"python scripts/prepare_tokens.py --train-file data/clean/pretrain.txt --train-mb {TRAIN_MB} "
    f"--val-file data/clean/heldout.txt --val-mb {VAL_MB} --out data/tokens --workers 4 {tok_args}")

## Step 4. Train the language models
Runs go to the GPUs in parallel (one per GPU), Mamba runs first because they are slower. Progress is printed every few minutes; each run's full log is saved in `results/lm/logs/`. Finished runs are skipped if you run this cell again.

In [ ]:
import shutil

os.makedirs("results/lm/logs", exist_ok=True)
queue = [(n, a) for a in ARCHS for n, _ in TOKS if not os.path.exists(f"results/lm/{n}__{a}.json")]
print("runs to do:", queue)


SHRINK = {}  # (tok, arch) -> how many times the batch was halved after running out of GPU memory


def cmd_for(tok, arch):
    # Mamba uses more memory in its scan: half the batch, two accumulation steps (same tokens per step)
    batch, accum = (8, 2) if arch == "mamba" else (16, 1)
    k = SHRINK.get((tok, arch), 0)
    batch, accum = max(1, batch >> k), accum << k     # same tokens per step, less memory
    return (f"python scripts/train_lm.py --tokens data/tokens/{tok} --arch {arch} --out results/lm "
            f"--seq-len {SEQ_LEN} --batch {batch} --grad-accum {accum} --seed {SEED} --eval-every 500")


def tail(path, n=1):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 3000))
            lines = [l for l in f.read().decode("utf-8", "replace").splitlines() if l.strip()]
            return lines[-n:]
    except FileNotFoundError:
        return []


slots = list(range(max(1, N_GPU)))
running, failed, t_start, last_print = {}, [], time.time(), 0
while queue or running:
    for g in slots:
        if g not in running and queue:
            tok, arch = queue.pop(0)
            log = f"results/lm/logs/{tok}__{arch}.log"
            env = dict(os.environ)
            if N_GPU:
                env["CUDA_VISIBLE_DEVICES"] = str(g)
            p = subprocess.Popen(cmd_for(tok, arch), shell=True, env=env,
                                 stdout=open(log, "w"), stderr=subprocess.STDOUT)
            running[g] = (p, tok, arch, log)
            print(f"[{(time.time() - t_start) / 60:6.1f} min] started {tok} / {arch} on GPU {g}", flush=True)
    for g, (p, tok, arch, log) in list(running.items()):
        if p.poll() is not None:
            ok = p.returncode == 0
            print(f"[{(time.time() - t_start) / 60:6.1f} min] {'finished' if ok else 'FAILED'} {tok} / {arch}: "
                  f"{tail(log)[-1] if tail(log) else ''}", flush=True)
            if ok:
                for f in (f"results/lm/{tok}__{arch}.json", log):
                    if os.path.exists(f):
                        shutil.copy(f, RUNS_OUT)
            if not ok:
                if "out of memory" in " ".join(tail(log, 40)).lower() and SHRINK.get((tok, arch), 0) < 3:
                    SHRINK[(tok, arch)] = SHRINK.get((tok, arch), 0) + 1
                    queue.insert(0, (tok, arch))
                    print(f"    out of GPU memory: retrying {tok}/{arch} with half the batch", flush=True)
                else:
                    failed.append((tok, arch))
                    print("\n".join(tail(log, 25)))
            del running[g]
    if time.time() - last_print > 300 and running:
        last_print = time.time()
        for g, (p, tok, arch, log) in running.items():
            print(f"    GPU {g} {tok}/{arch}: {(tail(log) or [''])[-1].strip()}", flush=True)
    time.sleep(10)
print("all runs done" + (f"; failed: {failed}" if failed else ""))

## Step 5. Results
**Bits per byte** is the headline: lower means the model predicts Nepali text better. Compare plain vs AksharaPiece within each architecture.

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

run("python scripts/summarize_lm.py --runs results/lm --out results/lm_summary")
display(Image("results/lm_summary.png"))

df = pd.read_csv("results/lm_summary.csv")
for arch in df["arch"].unique():
    for v in VOCABS:
        k = f"bpe_{v // 1000}k"
        a = df[(df.arch == arch) & (df.tokenizer == f"plain_{k}")]
        b = df[(df.arch == arch) & (df.tokenizer == f"akshara_{k}")]
        if len(a) and len(b):
            a, b = a.iloc[0], b.iloc[0]
            diff = 100 * (b.bpb / a.bpb - 1)
            verdict = "better" if diff < 0 else "worse"
            display(Markdown(f"**{arch}, {v // 1000}k:** AksharaPiece BPB {b.bpb:.4f} vs plain {a.bpb:.4f} "
                             f"→ {abs(diff):.2f}% {verdict}"))

## Step 6. Save the results
`aksharapiece_lm_results.zip` (in the **Output** panel) holds the summary table and chart, every run's JSON and log, and the trained tokenizers. Send it back to continue.

In [ ]:
import shutil

bundle = os.path.join(WORK, "lm_bundle")
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree("results", os.path.join(bundle, "results"))
shutil.copytree("data/tok", os.path.join(bundle, "tokenizers"))
for f in ("data/clean/stats.json",):
    shutil.copy(f, bundle)
for n, _ in TOKS:
    shutil.copy(f"data/tokens/{n}/meta.json", os.path.join(bundle, f"tokens_meta_{n}.json"))
shutil.make_archive(os.path.join(OUT, "aksharapiece_lm_results"), "zip", bundle)
shutil.rmtree(bundle)
print("Saved", os.path.join(OUT, "aksharapiece_lm_results.zip"))

## How to read the results

- **Bits per byte, plain vs AksharaPiece, same architecture:** the main result. A difference of about 0.5% or more is worth following up with more seeds; smaller differences may be noise from a single run.
- **Transformer vs Mamba gap:** research question 3 asks whether the tokenizer matters more for Mamba.
- **Bytes per token:** AksharaPiece needs slightly more tokens for the same text; bits per byte already accounts for that.
- Next steps after this run: repeat with 2 more seeds, add the 8k vocabulary, and run the downstream tasks.